In [1]:
import polars as pl

train = pl.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')

print(train.head(2))

shape: (2, 3)
┌──────────┬─────────────────────────────────┬──────────┐
│ id       ┆ prompt                          ┆ answer   │
│ ---      ┆ ---                             ┆ ---      │
│ str      ┆ str                             ┆ str      │
╞══════════╪═════════════════════════════════╪══════════╡
│ 00066667 ┆ In Alice's Wonderland, a secre… ┆ 10010111 │
│ 000b53cf ┆ In Alice's Wonderland, a secre… ┆ 01000011 │
└──────────┴─────────────────────────────────┴──────────┘


In [2]:
import kagglehub

# Download latest version
path = kagglehub.model_download("kienngx/nemotron-nano-30b-trained/triton/tinker-adapter")

print("Path to model files:", path)

Path to model files: /kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1


In [3]:
import shutil
import os
import re
import glob
import json
import zipfile as _zf
import torch

from safetensors import safe_open
from safetensors.torch import save_file

# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════

VARIANCE_RETAINED = 0.98
MIN_RANK_RATIO = 0.50
USE_FLOAT64_SVD = True

WORK_DIR = "/kaggle/working"

ORIGINAL_ADAPTER_NAME = "adapter_model.safetensors"
CONVERTED_ADAPTER_NAME = "converted_adapter_model.safetensors"

possible_adapter_paths = [
    "/kaggle/input/datasets/assiaben/adapter-v32-epoch-3/adapter",
    "/kaggle/input/adapter-v32-epoch-3/adapter",
    "/kaggle/input/adapter",
]

# ════════════════════════════════════════════════════════════════
# CLEAN OLD CORRUPTED FILES
# ════════════════════════════════════════════════════════════════

corrupted_files = [
    f"{WORK_DIR}/{CONVERTED_ADAPTER_NAME}",
]

for f in corrupted_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"🧹 Removed old file: {f}")

# ════════════════════════════════════════════════════════════════
# FIND ADAPTER
# ════════════════════════════════════════════════════════════════

ADAPTER_PATH = None

for path in possible_adapter_paths:

    config_exists = os.path.exists(
        os.path.join(path, "adapter_config.json")
    )

    tensor_exists = os.path.exists(
        os.path.join(path, ORIGINAL_ADAPTER_NAME)
    )

    if config_exists and tensor_exists:
        ADAPTER_PATH = path
        print(f"✅ Found adapter at: {path}")
        break

# fallback search
if ADAPTER_PATH is None:

    print("🔍 Searching entire /kaggle/input...")

    for root, dirs, files in os.walk("/kaggle/input/"):

        if (
            "adapter_config.json" in files
            and ORIGINAL_ADAPTER_NAME in files
        ):
            ADAPTER_PATH = root
            print(f"✅ Found adapter at: {root}")
            break

if ADAPTER_PATH is None:
    raise FileNotFoundError(
        "❌ Could not find adapter files"
    )

# ════════════════════════════════════════════════════════════════
# COPY ADAPTER TO WORKING
# ════════════════════════════════════════════════════════════════

working_config = f"{WORK_DIR}/adapter_config.json"
working_adapter = f"{WORK_DIR}/{ORIGINAL_ADAPTER_NAME}"

print("📦 Copying adapter files...")

shutil.copy2(
    os.path.join(
        ADAPTER_PATH,
        "adapter_config.json",
    ),
    working_config,
)

shutil.copy2(
    os.path.join(
        ADAPTER_PATH,
        ORIGINAL_ADAPTER_NAME,
    ),
    working_adapter,
)

print("✅ Adapter copied")

# ════════════════════════════════════════════════════════════════
# FIX CONFIG
# ════════════════════════════════════════════════════════════════

with open(working_config, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = (
    "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
)

cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

cfg["target_modules"] = [
    "k_proj",
    "o_proj",
    "in_proj",
    "q_proj",
    "up_proj",
    "v_proj",
    "down_proj",
    "out_proj",
    "lm_head",
]

with open(working_config, "w") as f:
    json.dump(cfg, f, indent=2)

print("✅ Config fixed")

# ════════════════════════════════════════════════════════════════
# LOAD MODEL SHAPES
# ════════════════════════════════════════════════════════════════

import kagglehub

MODEL_PATH = (
    "/kaggle/input/models/metric/"
    "nemotron-3-nano-30b-a3b-bf16/"
    "transformers/default/1"
)

if not os.path.exists(MODEL_PATH):

    print("⬇️ Downloading model...")

    MODEL_PATH = kagglehub.model_download(
        "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    )

print(f"✅ Model path: {MODEL_PATH}")

model_files = glob.glob(
    f"{MODEL_PATH}/*.safetensors"
)

if not model_files:
    raise FileNotFoundError(
        f"❌ No model safetensors found"
    )

model_key_shapes = {}

print("📦 Reading model shapes...")

for model_file in model_files:

    print(f"   → {os.path.basename(model_file)}")

    with safe_open(
        model_file,
        framework="pt",
        device="cpu",
    ) as f:

        for key in f.keys():

            tensor_slice = f.get_slice(key)

            model_key_shapes[key] = tuple(
                tensor_slice.get_shape()
            )

print(
    f"✅ Loaded {len(model_key_shapes)} model keys"
)

# ════════════════════════════════════════════════════════════════
# LOAD ADAPTER
# ════════════════════════════════════════════════════════════════

print("📦 Loading adapter tensors...")

adapter_tensors = {}

with safe_open(
    working_adapter,
    framework="pt",
    device="cpu",
) as f:

    for key in f.keys():
        adapter_tensors[key] = f.get_tensor(key)

print(
    f"✅ Loaded {len(adapter_tensors)} adapter tensors"
)

# ════════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════════

def trained_adapter_key_rename(
    key_name: str
) -> str:

    return key_name.replace(
        "base_model.model.model",
        "base_model.model.backbone",
    )

# ════════════════════════════════════════════════════════════════
# COLLECT BASE NAMES
# ════════════════════════════════════════════════════════════════

base_names = set()

for key in adapter_tensors:

    base = re.sub(
        r"\.lora_[AB]\.weight$",
        "",
        key,
    )

    base_names.add(base)

print(
    f"✅ Collected {len(base_names)} base names"
)

# ════════════════════════════════════════════════════════════════
# IDENTIFY MAMBA MERGES
# ════════════════════════════════════════════════════════════════

mamba_merge_layers = {}

for base in base_names:

    for proj in ("gate_proj", "x_proj"):

        if f".{proj}" in base:

            layer_path = base.rsplit(
                f".{proj}",
                1,
            )[0]

            mamba_merge_layers.setdefault(
                layer_path,
                {},
            )[proj] = base

mamba_merge_bases = set()

for projs in mamba_merge_layers.values():
    mamba_merge_bases.update(projs.values())

print(
    f"✅ Mamba merge layers: "
    f"{len(mamba_merge_layers)}"
)

# ════════════════════════════════════════════════════════════════
# BUILD OUTPUT TENSORS
# ════════════════════════════════════════════════════════════════

tensors = {}

print("⚙️ Processing tensors...")

for base in sorted(base_names):

    if "base_layer" in base:
        continue

    lora_A = adapter_tensors.get(
        f"{base}.lora_A.weight"
    )

    lora_B = adapter_tensors.get(
        f"{base}.lora_B.weight"
    )

    if lora_A is None or lora_B is None:
        continue

    renamed = trained_adapter_key_rename(base)

    # skip empty experts
    if (
        ".experts.w3" in base
        and lora_A.numel() == 0
    ):
        continue

    # skip mamba merge tensors
    if base in mamba_merge_bases:
        continue

    # ════════════════════════════════════════════════════════
    # EXPERT UNFUSING
    # ════════════════════════════════════════════════════════

    if (
        ".experts.w1" in base
        or ".experts.w2" in base
    ):

        if lora_A.shape[0] == 1:

            lora_A = lora_A.expand(
                lora_B.shape[0],
                -1,
                -1,
            ).contiguous()

        elif lora_B.shape[0] == 1:

            lora_B = lora_B.expand(
                lora_A.shape[0],
                -1,
                -1,
            ).contiguous()

        num_experts = lora_A.shape[0]

        proj_name = (
            "up_proj"
            if ".w1" in base
            else "down_proj"
        )

        for i in range(num_experts):

            exp_renamed = re.sub(
                r"\.experts\.w[12]",
                f".experts.{i}.{proj_name}",
                renamed,
            )

            tensors[
                f"{exp_renamed}.lora_A.weight"
            ] = lora_A[i].contiguous()

            tensors[
                f"{exp_renamed}.lora_B.weight"
            ] = lora_B[i].contiguous()

        continue

    # normal tensors
    tensors[
        f"{renamed}.lora_A.weight"
    ] = lora_A.contiguous()

    tensors[
        f"{renamed}.lora_B.weight"
    ] = lora_B.contiguous()

# ════════════════════════════════════════════════════════════════
# MAMBA MERGE + STABLE SVD
# ════════════════════════════════════════════════════════════════

print("⚙️ Processing Mamba merges...")

for layer_path, projs in sorted(
    mamba_merge_layers.items()
):

    if (
        "gate_proj" not in projs
        or "x_proj" not in projs
    ):
        continue

    renamed_layer = trained_adapter_key_rename(
        layer_path
    )

    in_proj_base = (
        f"{renamed_layer}.in_proj"
    )

    model_in_proj_key = (
        renamed_layer.replace(
            "base_model.model.",
            "",
        )
        + ".in_proj.weight"
    )

    if model_in_proj_key not in model_key_shapes:
        continue

    in_proj_dim = model_key_shapes[
        model_in_proj_key
    ][0]

    gate_A = adapter_tensors[
        f"{projs['gate_proj']}.lora_A.weight"
    ].float()

    gate_B = adapter_tensors[
        f"{projs['gate_proj']}.lora_B.weight"
    ].float()

    x_A = adapter_tensors[
        f"{projs['x_proj']}.lora_A.weight"
    ].float()

    x_B = adapter_tensors[
        f"{projs['x_proj']}.lora_B.weight"
    ].float()

    rank = gate_A.shape[0]

    # ════════════════════════════════════════════════════════
    # BUILD MERGED MATRIX
    # ════════════════════════════════════════════════════════

    A_cat = torch.cat(
        [gate_A, x_A],
        dim=0,
    )

    B_block = torch.zeros(
        in_proj_dim,
        2 * rank,
        dtype=torch.float32,
    )

    B_block[
        :gate_B.shape[0],
        :rank,
    ] = gate_B

    B_block[
        gate_B.shape[0]:
        gate_B.shape[0] + x_B.shape[0],
        rank:,
    ] = x_B

    merged_matrix = B_block @ A_cat

    if USE_FLOAT64_SVD:
        merged_matrix = merged_matrix.to(
            torch.float64
        )

    try:

        # ════════════════════════════════════════════════
        # SVD
        # ════════════════════════════════════════════════

        U, S, Vh = torch.linalg.svd(
            merged_matrix,
            full_matrices=False,
        )

        # ════════════════════════════════════════════════
        # ENERGY-BASED RANK SELECTION
        # ════════════════════════════════════════════════

        energy = S.pow(2)

        cumsum = (
            torch.cumsum(
                energy,
                dim=0,
            )
            / energy.sum()
        )

        k = (
            torch.searchsorted(
                cumsum,
                VARIANCE_RETAINED,
            ).item()
            + 1
        )

        min_rank = max(
            1,
            int(rank * MIN_RANK_RATIO),
        )

        k = max(k, min_rank)
        k = min(k, rank)

        print(
            f"🧠 {layer_path}: "
            f"kept {k}/{rank}"
        )

        # ════════════════════════════════════════════════
        # REBUILD FACTORS
        # ════════════════════════════════════════════════

        new_B = (
            U[:, :k]
            * S[:k].unsqueeze(0)
        )

        new_A = Vh[:k, :]

        new_B = (
            new_B.float()
            .contiguous()
        )

        new_A = (
            new_A.float()
            .contiguous()
        )

        tensors[
            f"{in_proj_base}.lora_A.weight"
        ] = new_A

        tensors[
            f"{in_proj_base}.lora_B.weight"
        ] = new_B

    except Exception as e:

        print(
            f"⚠️ Failed merge: {layer_path}"
        )

        print(e)

# ════════════════════════════════════════════════════════════════
# SAVE CONVERTED ADAPTER
# ════════════════════════════════════════════════════════════════

converted_adapter_path = (
    f"{WORK_DIR}/{CONVERTED_ADAPTER_NAME}"
)

print("💾 Saving converted adapter...")

save_file(
    tensors,
    converted_adapter_path,
)

print(
    f"✅ Saved converted adapter "
    f"with {len(tensors)} tensors"
)

# ════════════════════════════════════════════════════════════════
# BUILD SUBMISSION ZIP
# ════════════════════════════════════════════════════════════════

submission_zip = (
    f"{WORK_DIR}/submission.zip"
)

print("📦 Creating submission zip...")

with _zf.ZipFile(
    submission_zip,
    "w",
    _zf.ZIP_DEFLATED,
) as zf:

    # config
    zf.write(
        working_config,
        "adapter_config.json",
    )

    print("✅ Added config")

    # IMPORTANT:
    # rename converted adapter back to expected filename
    zf.write(
        converted_adapter_path,
        "adapter_model.safetensors",
    )

    print("✅ Added converted adapter")

# ════════════════════════════════════════════════════════════════
# FINAL CHECK
# ════════════════════════════════════════════════════════════════

if os.path.exists(submission_zip):

    size_mb = (
        os.path.getsize(submission_zip)
        / 1024
        / 1024
    )

    print("\n🎯 submission.zip ready!")
    print(f"📦 Size: {size_mb:.2f} MB")

else:
    print("❌ submission.zip missing")

🔍 Searching entire /kaggle/input...
✅ Found adapter at: /kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1
📦 Copying adapter files...
✅ Adapter copied
✅ Config fixed
✅ Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
📦 Reading model shapes...
   → model-00012-of-00013.safetensors
   → model-00004-of-00013.safetensors
   → model-00010-of-00013.safetensors
   → model-00008-of-00013.safetensors
   → model-00011-of-00013.safetensors
   → model-00002-of-00013.safetensors
   → model-00005-of-00013.safetensors
   → model-00003-of-00013.safetensors
   → model-00009-of-00013.safetensors
   → model-00007-of-00013.safetensors
   → model-00006-of-00013.safetensors
   → model-00001-of-00013.safetensors
   → model-00013-of-00013.safetensors
✅ Loaded 6243 model keys
📦 Loading adapter tensors...
✅ Loaded 12010 adapter tensors
✅ Collected 6005 base names
✅ Mamba merge layers: 0
⚙️ Processing tensors...
⚙️ Processing Mamba merges...
